# AI4I 분할 v2 — AutoGluon 모델별 · 앙상블 조합별 비교

- 학습 = train, 고르기 = val(`score_val`), 마지막 확인 = test(`score_test`, 고를 때 보지 않음). demo 몫은 쓰지 않음
- 데이터: `../csv/ai4i-{train,val,test}.csv` (README 참고)
- 설치: 같은 폴더 `requirements.txt` (Python 3.12 가상환경 권장)

## 1. 설정

In [1]:
from itertools import combinations
from pathlib import Path

import pandas as pd
from autogluon.tabular import TabularPredictor

DATA = Path("../csv")        # 이 노트북이 examples/ 폴더에 있을 때
TARGET = "failure_type"      # "failure_type"(NONE · TWF · HDF · PWF · OSF) 또는 "Machine failure"(0/1)
ADD_FEATURES = True          # 온도 차 · 출력 칸을 더함(Pack도 같은 값을 계산함)
TIME_LIMIT = 600             # 초
TOP_K = 4                    # 조합을 만들 상위 모델 수(val 기준)

DROP = ["UDI", "Product ID", "Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]
assert (DATA / "ai4i-train.csv").exists(), "DATA 경로를 csv 폴더로 맞춰 주세요"

## 2. 데이터 읽기 · 정답 칸 만들기

In [2]:
def load(name):
    df = pd.read_csv(DATA / f"ai4i-{name}.csv")
    if TARGET == "failure_type":
        # 한 행에 고장 유형이 둘 이상이면 TWF > HDF > PWF > OSF 순으로 하나(train 14행). RNF(무작위 고장)만 있는 행은 NONE
        df[TARGET] = "NONE"
        for t in ["OSF", "PWF", "HDF", "TWF"]:
            df.loc[df[t] == 1, TARGET] = t
    else:
        df[TARGET] = df["Machine failure"]
    if ADD_FEATURES:
        df["temp_diff_k"] = df["Process temperature [K]"] - df["Air temperature [K]"]
        df["power_w"] = df["Torque [Nm]"] * df["Rotational speed [rpm]"] * 2 * 3.141592653589793 / 60
    return df.drop(columns=[c for c in DROP if c != TARGET])


train, val, test = load("train"), load("val"), load("test")
pd.DataFrame({k: d[TARGET].value_counts() for k, d in [("train", train), ("val", val), ("test", test)]})

,train,val,test
failure_type,,,
HDF,49,15,16
NONE,5561,1852,1855
OSF,46,17,15
PWF,53,19,19
TWF,26,9,9


## 3. 학습 (모델별)

In [3]:
metric = "f1_macro" if TARGET == "failure_type" else "f1"
predictor = TabularPredictor(
    label=TARGET,
    eval_metric=metric,
    sample_weight="balance_weight",   # 고장 행이 적어서 클래스 비율로 무게를 맞춤
    path="ag-ai4i",
).fit(
    train_data=train,
    tuning_data=val,
    presets="medium_quality",
    time_limit=TIME_LIMIT,
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.15
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.6.0: Mon Jul 14 11:30:30 PDT 2025; root:xnu-11417.140.69~1/RELEASE_ARM64_T6020
CPU Count:          12
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       6.84 GB / 32.00 GB (21.4%)
Disk Space Avail:   169.37 GB / 926.35 GB (18.3%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Using predefined sample weighting strategy: balance_weight. Evaluation metrics will ignore sample weights, specify weight_evaluation=True to instead report weighted metrics.
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/insujo/Documents/repo/orchestration/data/ai4i

## 4. 앙상블 조합 만들기 (val 상위 TOP_K로 2 · 3개 조합)

In [4]:
lb = predictor.leaderboard(test, silent=True)
singles = lb[~lb["model"].str.startswith("WeightedEnsemble")].sort_values("score_val", ascending=False)
top = singles["model"].head(TOP_K).tolist()
print("조합에 쓸 모델:", top)
for r in (2, 3):
    for combo in combinations(top, r):
        suffix = "_" + "+".join(m.split("_")[0] for m in combo)
        predictor.fit_weighted_ensemble(base_models=list(combo), name_suffix=suffix)

Fitting model: WeightedEnsemble_L2_XGBoost+LightGBMLarge ...
	Fitting 1 model on all data | Fitting with cpus=12, gpus=0, mem=0.0/6.0 GB
	Ensemble Weights: {'LightGBMLarge': 0.6, 'XGBoost': 0.4}
	0.8034	 = Validation score   (f1_macro)
	0.04s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: WeightedEnsemble_L2_XGBoost+LightGBM ...
	Fitting 1 model on all data | Fitting with cpus=12, gpus=0, mem=0.0/6.0 GB
	Ensemble Weights: {'XGBoost': 1.0}
	0.7996	 = Validation score   (f1_macro)
	0.04s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: WeightedEnsemble_L2_XGBoost+CatBoost ...
	Fitting 1 model on all data | Fitting with cpus=12, gpus=0, mem=0.0/6.0 GB
	Ensemble Weights: {'XGBoost': 1.0}
	0.7996	 = Validation score   (f1_macro)
	0.04s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: WeightedEnsemble_L2_LightGBMLarge+LightGBM ...
	Fitting 1 model on all data | Fitting with cpus=12, gpus=0, mem=0.0/6.0 GB
	Ensemble Weights: {'LightGBMLarge': 

조합에 쓸 모델: ['XGBoost', 'LightGBMLarge', 'LightGBM', 'CatBoost']


	Ensemble Weights: {'LightGBMLarge': 1.0}
	0.7838	 = Validation score   (f1_macro)
	0.04s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: WeightedEnsemble_L2_LightGBM+CatBoost ...
	Fitting 1 model on all data | Fitting with cpus=12, gpus=0, mem=0.0/6.0 GB
	Ensemble Weights: {'LightGBM': 1.0}
	0.7781	 = Validation score   (f1_macro)
	0.04s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: WeightedEnsemble_L2_XGBoost+LightGBMLarge+LightGBM ...
	Fitting 1 model on all data | Fitting with cpus=12, gpus=0, mem=0.0/6.0 GB
	Ensemble Weights: {'LightGBMLarge': 0.6, 'XGBoost': 0.4}
	0.8034	 = Validation score   (f1_macro)
	0.06s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: WeightedEnsemble_L2_XGBoost+LightGBMLarge+CatBoost ...
	Fitting 1 model on all data | Fitting with cpus=12, gpus=0, mem=0.0/6.0 GB
	Ensemble Weights: {'LightGBMLarge': 0.6, 'XGBoost': 0.4}
	0.8034	 = Validation score   (f1_macro)
	0.06s	 = Training   runtime
	0.0s	 = Validat

## 5. 비교표 — val 점수로 고르고, test는 확인만

In [5]:
lb = predictor.leaderboard(test, extra_metrics=["balanced_accuracy", "accuracy"], silent=True)
lb = lb.sort_values("score_val", ascending=False)
cols = ["model", "score_val", "score_test", "balanced_accuracy", "accuracy", "pred_time_test", "fit_time"]
lb[cols].to_csv("ag-ai4i-leaderboard.csv", index=False)
lb[cols]

,model,score_val,score_test,balanced_accuracy,accuracy,pred_time_test,fit_time
6,WeightedEnsemble_L2,0.805510,0.725363,0.776919,0.975967,0.123930,32.683630
7,WeightedEnsemble_L2_XGBoost+LightGBMLarge+LightGBM,0.803350,0.719750,0.766501,0.975967,0.022538,29.134318
8,WeightedEnsemble_L2_XGBoost+LightGBMLarge,0.803350,0.719750,0.766501,0.975967,0.022608,29.115669
9,WeightedEnsemble_L2_XGBoost+LightGBMLarge+CatBoost,0.803350,0.719750,0.766501,0.975967,0.022661,29.132536
2,WeightedEnsemble_L2_LightGBMLarge+LightGBM+CatBoost,0.800050,0.746444,0.746003,0.983804,0.044412,63.249773
11,XGBoost,0.799605,0.715951,0.775949,0.971264,0.015522,2.355970
12,WeightedEnsemble_L2_XGBoost+LightGBM+CatBoost,0.799605,0.715951,0.775949,0.971264,0.016068,2.411251
14,WeightedEnsemble_L2_XGBoost+CatBoost,0.799605,0.715951,0.775949,0.971264,0.016218,2.393074
13,WeightedEnsemble_L2_XGBoost+LightGBM,0.799605,0.715951,0.775949,0.971264,0.016095,2.393693
3,WeightedEnsemble_L2_LightGBMLarge+LightGBM,0.794630,0.744944,0.745788,0.982759,0.031417,34.542947


In [6]:
best = lb.iloc[0]["model"]
print(f"val 1등: {best} — test {lb.iloc[0]['score_test']:.4f}")
if best.startswith("WeightedEnsemble"):
    print("앙상블이 쓴 모델:", list(predictor.info()["model_info"][best].get("children_info", {}).keys()))

val 1등: WeightedEnsemble_L2 — test 0.7254
앙상블이 쓴 모델: ['S1F1']


## 6. 1등 모델의 test 상세 (클래스별)

In [7]:
y_pred = predictor.predict(test, model=best)
print(pd.crosstab(test[TARGET], y_pred, rownames=["정답"], colnames=["예측"]))
predictor.evaluate(test, model=best, detailed_report=True)["classification_report"]

예측    HDF  NONE  OSF  PWF  TWF
정답                            
HDF    16     0    0    0    0
NONE    4  1821    5    2   23
OSF     0     4   11    0    0
PWF     0     1    0   18    0
TWF     0     7    0    0    2


{'HDF': {'precision': 0.8,
  'recall': 1.0,
  'f1-score': 0.8888888888888888,
  'support': 16.0},
 'NONE': {'precision': 0.9934533551554828,
  'recall': 0.9816711590296496,
  'f1-score': 0.987527114967462,
  'support': 1855.0},
 'OSF': {'precision': 0.6875,
  'recall': 0.7333333333333333,
  'f1-score': 0.7096774193548387,
  'support': 15.0},
 'PWF': {'precision': 0.9,
  'recall': 0.9473684210526315,
  'f1-score': 0.9230769230769231,
  'support': 19.0},
 'TWF': {'precision': 0.08,
  'recall': 0.2222222222222222,
  'f1-score': 0.11764705882352941,
  'support': 9.0},
 'accuracy': 0.9759665621734587,
 'macro avg': {'precision': 0.6921906710310965,
  'recall': 0.7769190271275673,
  'f1-score': 0.7253634810223285,
  'support': 1914.0},
 'weighted avg': {'precision': 0.9842155035597808,
  'recall': 0.9759665621734587,
  'f1-score': 0.9797949147570847,
  'support': 1914.0}}

## 메모
- 고르는 기준은 val. test로 고르면 마지막 평가가 부풀려짐
- val을 학습에 섞지 않음 — C1-a15 평가 카드(val 30 · test 30)의 출처
- 최종 모델은 ONNX로 넘김(D-237). 앙상블은 ONNX로 바꾸기 어려우니, 넘길 후보는 단일 트리 모델 등 바꿀 수 있는 것으로(추정)